# 01 · Data Exploration
**Flight Analytics Platform** — OpenSky Network real-time data

This notebook demonstrates:
- Connecting to the PostgreSQL data warehouse
- Loading and inspecting raw + staging layer data
- Descriptive statistics and missing-value analysis
- Geographic visualisation with Plotly

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import psycopg2
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.2f}'.format)

# Connection helper
def get_conn():
    return psycopg2.connect(
        dbname  = os.getenv('POSTGRES_DB',       'flightdw'),
        user    = os.getenv('POSTGRES_USER',     'flightuser'),
        password= os.getenv('POSTGRES_PASSWORD', 'flightpass123'),
        host    = os.getenv('POSTGRES_HOST',     'postgres'),
        port    = os.getenv('POSTGRES_PORT',     '5432'),
    )

def sql(query, params=None):
    with get_conn() as conn:
        return pd.read_sql(query, conn, params=params)

print('✅ Connected to PostgreSQL')

## 1. Table Row Counts

In [ ]:
counts = sql("""
    SELECT 'raw.flight_events_raw'     AS tbl, COUNT(*) AS rows FROM raw.flight_events_raw
    UNION ALL
    SELECT 'staging.flight_events_clean',       COUNT(*) FROM staging.flight_events_clean
    UNION ALL
    SELECT 'main.flight_analytics',             COUNT(*) FROM main.flight_analytics
    UNION ALL
    SELECT 'main.country_traffic_summary',      COUNT(*) FROM main.country_traffic_summary
    UNION ALL
    SELECT 'main.hourly_flight_counts',         COUNT(*) FROM main.hourly_flight_counts
""")
display(counts.style.format({'rows': '{:,}'}))

## 2. Load Staging Data

In [ ]:
df = sql("""
    SELECT *
    FROM staging.flight_events_clean
    WHERE ingested_at >= NOW() - INTERVAL '1 hour'
    ORDER BY ingested_at DESC
    LIMIT 5000
""")
print(f'Shape: {df.shape}')
df.head()

## 3. Descriptive Statistics

In [ ]:
numeric_cols = ['longitude','latitude','baro_altitude','velocity_kmh','true_track','vertical_rate']
df[numeric_cols].describe().round(2)

## 4. Missing Values

In [ ]:
miss = df.isnull().sum().reset_index()
miss.columns = ['column', 'missing']
miss['pct_missing'] = (miss['missing'] / len(df) * 100).round(1)
miss = miss.sort_values('pct_missing', ascending=False)

fig = px.bar(
    miss[miss['pct_missing'] > 0],
    x='pct_missing', y='column', orientation='h',
    title='Missing Values (%)',
    template='plotly_dark',
    labels={'pct_missing': '% Missing', 'column': 'Column'},
)
fig.show()

## 5. Flights by Origin Country

In [ ]:
top_countries = (
    df.groupby('origin_country')['icao24'].nunique()
    .sort_values(ascending=False).head(20).reset_index()
)
top_countries.columns = ['country', 'unique_aircraft']

fig = px.bar(
    top_countries, x='unique_aircraft', y='country',
    orientation='h', template='plotly_dark',
    title='Top 20 Countries by Unique Aircraft',
    color='unique_aircraft', color_continuous_scale='Viridis',
)
fig.update_layout(showlegend=False, coloraxis_showscale=False)
fig.show()

## 6. Geographic Scatter Map

In [ ]:
map_df = df.dropna(subset=['longitude','latitude'])

fig = px.scatter_geo(
    map_df,
    lat='latitude', lon='longitude',
    color='velocity_kmh',
    color_continuous_scale='Plasma',
    hover_name='callsign',
    hover_data={'origin_country': True, 'baro_altitude': ':.0f'},
    title='Live Flight Positions',
    template='plotly_dark',
    opacity=0.7,
)
fig.update_geos(
    showcoastlines=True, coastlinecolor='#444',
    showland=True, landcolor='#1a1a2e',
    showocean=True, oceancolor='#0d0d1a',
    projection_type='natural earth',
)
fig.show()

## 7. Altitude Distribution by Level

In [ ]:
alt_counts = df['altitude_level'].value_counts().reset_index()
alt_counts.columns = ['level', 'count']

colors = {
    'ground':'#ef4444','low':'#f97316',
    'medium':'#eab308','high':'#22c55e',
    'very_high':'#3b82f6','unknown':'#6b7280'
}

fig = go.Figure(go.Pie(
    labels=alt_counts['level'],
    values=alt_counts['count'],
    hole=0.5,
    marker=dict(colors=[colors.get(l,'#888') for l in alt_counts['level']]),
))
fig.update_layout(title='Altitude Level Distribution', template='plotly_dark')
fig.show()

## 8. Velocity vs Altitude Scatter
Expect a positive correlation — higher altitude → higher speed.

In [ ]:
scatter_df = df.dropna(subset=['baro_altitude','velocity_kmh'])
scatter_df = scatter_df[scatter_df['velocity_kmh'] > 50]   # filter ground traffic

fig = px.scatter(
    scatter_df.sample(min(2000, len(scatter_df))),
    x='velocity_kmh', y='baro_altitude',
    color='origin_country',
    opacity=0.5,
    title='Velocity vs Altitude',
    labels={'velocity_kmh': 'Speed (km/h)', 'baro_altitude': 'Altitude (m)'},
    template='plotly_dark',
)
fig.update_layout(showlegend=False)
fig.show()